# 08 - Supplementary baseline characteristics

Cohort exclusion counts, distributions of binary and continuous covariates, and the extended baseline characteristics table (including ethnicity, blood pressure, polypharmacy and PPI use).

**Manuscript items:** Fig S1 (exclusions); Figs S2, S3; Table S8

**Inputs:** derived files from 01, 02 and 07 (polypharmacy groups); CPRD Observation and DrugIssue extracts; ethnicity code list

**Outputs:** tables displayed in the notebook; `SUPP_FIGURES_DIR/distribution_of_binary_variables.png`, `SUPP_FIGURES_DIR/fig1_distribution_of_continuous_variables.png`

*Derived from the original analysis notebook `4_cohort_characteristics_DDI_sup.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

# Cohort characteristics — DDI supplementary tables

Supplementary characteristics tables for the DOAC drug–drug interaction cohort.
Each patient characteristic is handled in its own step: load once, define shared
helpers once, then build one table per characteristic (binary comorbidities,
continuous measurements, age, calendar year, blood pressure, ethnicity,
polypharmacy, PPI use, and exposure/follow-up).

## Setup

In [ ]:
import os
import zipfile
import warnings
from multiprocessing import Pool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, mannwhitneyu

warnings.filterwarnings("ignore")

# supplementary figures/tables are written here
os.makedirs(SUPP_DIR, exist_ok=True)

In [ ]:
n_processes = 60

## 1. Load DOAC cohorts

The major-interaction file is read once as the *unfiltered* cohort
(`major_doac_exposure`), which is reused later for the exclusion counts and for
mapping study-entry dates and patient membership. The analytic (inclusion-
applied) frames are derived from it and from the minor/all files.

In [ ]:
def read_cohort_raw(path):
    """Read a cohort export and coerce the shared date / age columns."""
    df = pd.read_excel(path, dtype=str)
    df["age_at_study_start"] = df["age_at_study_start"].astype(int)
    for col in ["study_start_date", "study_end_date", "event_date"]:
        df[col] = pd.to_datetime(df[col])
    return df


def apply_inclusion(df):
    """Apply the study inclusion criteria and keep the analysis columns."""
    keep = (
        (df["age_at_study_start"] >= 18)
        & (df["atleast_1y_med_hist_flag"] == "1")
        & (df["taken_warfarin_1w_before"] == "0")
        & (df["taken_warfarin_during_study"] == "0")
    )
    cols = ["patid", "study_start_date", "study_end_date", "event_date"]
    return df.loc[keep, cols].copy()


# Unfiltered major cohort (reused for exclusions and date/membership mapping).
major_doac_exposure = read_cohort_raw(os.path.join(DERIVED_DIR, "cohort_DDI_major.xlsx"))

# Analytic cohorts.
serious_doac_exposure = apply_inclusion(major_doac_exposure)
minor_doac_exposure = apply_inclusion(read_cohort_raw(os.path.join(DERIVED_DIR, "cohort_DDI_minor.xlsx")))
all_doac_exposure = apply_inclusion(read_cohort_raw(os.path.join(DERIVED_DIR, "cohort_DDI_all.xlsx")))

for label, frame in [("Serious", serious_doac_exposure),
                     ("Minor", minor_doac_exposure),
                     ("All", all_doac_exposure)]:
    n = len(frame)
    n_event = frame["event_date"].notna().sum()
    print(f"{label:8s} cohort: {n:,} patients, {n_event:,} with a {label.lower()} event")

## 2. Bleeding-outcome patient sets

Patients are split into those with vs without an event for each outcome
definition. `Overall` (and therefore the denominator `total_patients`) is the
serious cohort, which contains every eligible patient.

In [ ]:
total_patients = serious_doac_exposure["patid"].nunique()


def event_patids(frame):
    """(with-event, without-event) patid arrays for a cohort frame."""
    with_event = frame.loc[frame["event_date"].notna(), "patid"].unique()
    without_event = frame.loc[frame["event_date"].isna(), "patid"].unique()
    return with_event, without_event


pats_with_serious_bleeding_events, pats_with_no_serious_bleeding_events = event_patids(serious_doac_exposure)
pats_with_minor_bleeding_events, pats_with_no_minor_bleeding_events = event_patids(minor_doac_exposure)
pats_with_all_bleeding_events, pats_with_no_all_bleeding_events = event_patids(all_doac_exposure)

for label, with_ev in [("serious", pats_with_serious_bleeding_events),
                       ("minor", pats_with_minor_bleeding_events),
                       ("all", pats_with_all_bleeding_events)]:
    pct = len(with_ev) / total_patients * 100
    print(f"Patients with {label} bleeding events: {len(with_ev):,} ({pct:.2f}%)")

## 3. Cohort exclusions

Mutually-exclusive exclusion counts from the unfiltered major cohort, applied in
order: age < 18, then < 1 year of prior history, then warfarin in the week
before, then warfarin during follow-up.

In [ ]:
pats_age_less_than_18 = major_doac_exposure.loc[
    major_doac_exposure["age_at_study_start"] < 18, "patid"
].unique()

pats_with_less_than_1y_med_hist = major_doac_exposure.loc[
    (major_doac_exposure["atleast_1y_med_hist_flag"] == "0")
    & ~major_doac_exposure["patid"].isin(pats_age_less_than_18),
    "patid",
].unique()

pats_with_warfarin_1w_before = major_doac_exposure.loc[
    (major_doac_exposure["taken_warfarin_1w_before"] == "1")
    & ~major_doac_exposure["patid"].isin(pats_age_less_than_18)
    & ~major_doac_exposure["patid"].isin(pats_with_less_than_1y_med_hist),
    "patid",
].unique()

pats_with_warfarin_during_study = major_doac_exposure.loc[
    (major_doac_exposure["taken_warfarin_during_study"] == "1")
    & ~major_doac_exposure["patid"].isin(pats_age_less_than_18)
    & ~major_doac_exposure["patid"].isin(pats_with_less_than_1y_med_hist)
    & ~major_doac_exposure["patid"].isin(pats_with_warfarin_1w_before),
    "patid",
].unique()

print(f"Age < 18: {len(pats_age_less_than_18):,}")
print(f"< 1 year medical history: {len(pats_with_less_than_1y_med_hist):,}")
print(f"Warfarin 1 week before: {len(pats_with_warfarin_1w_before):,}")
print(f"Warfarin during study: {len(pats_with_warfarin_during_study):,}")
print(f"Total excluded: {len(pats_age_less_than_18) + len(pats_with_less_than_1y_med_hist) + len(pats_with_warfarin_1w_before) + len(pats_with_warfarin_during_study):,}")

## 4. Patient demographics & comorbidities

Load patient sex, load the comorbidity table restricted to the major cohort,
coerce the binary/continuous columns to numeric, then rename to display labels.

In [ ]:
pats_data = pd.read_csv(PATIENT_FILE, sep=",", dtype=str)
pats_data["sex_m"] = np.where(pats_data["gender"] == "1", 1, 0)
pats_data["sex_f"] = np.where(pats_data["gender"] == "2", 1, 0)

In [ ]:
binary_cols = [
    "hypertension", "diabetes", "cancer", "liver_disease",
    "major_bled_hist", "alcohol", "anemia",
    "heart_failure", "chd", "angina", "myocardial_infraction",
    "atrial_fibrillation", "venous_thromboembolism", "smoking", "sex_m", "sex_f",
]
continuous_cols = [
    "age_at_study_start", "bmi_value", "ckd_value",
    "hemoglobin_value", "platelet_count", "cholesterol_total", "cholesterol_ldl",
    "cholesterol_hdl", "systolic_bp", "diastolic_bp", "pulse_rate",
]

In [ ]:
pats_comorbidities = pd.read_excel(os.path.join(DERIVED_DIR, "pats_comorbidities_DDI.xlsx"), dtype=str)
pats_comorbidities = pats_comorbidities[pats_comorbidities["patid"].isin(major_doac_exposure["patid"])]

pats_comorbidities["sex_m"] = pats_comorbidities["patid"].map(pats_data.set_index("patid")["sex_m"])
pats_comorbidities["sex_f"] = pats_comorbidities["patid"].map(pats_data.set_index("patid")["sex_f"])

for col in continuous_cols + binary_cols:
    pats_comorbidities[col] = pd.to_numeric(pats_comorbidities[col], errors="coerce")

print(f"{pats_comorbidities['patid'].nunique():,} patients in the comorbidity table")

In [ ]:
pats_comorbidities.rename(columns={
    "major_bled_hist": "Serious Bleeding History", "hypertension": "Hypertension",
    "diabetes": "Diabetes", "heart_failure": "Heart Failure",
    "cancer": "Cancer", "liver_disease": "Liver Disease",
    "chronic_kidney_disease": "Chronic Kidney Disease",
    "stroke": "Stroke", "atrial_fibrillation": "Atrial Fibrillation",
    "chd": "Coronary Heart Disease", "angina": "Angina",
    "myocardial_infraction": "Myocardial Infarction",
    "smoking": "Smoking Status", "venous_thromboembolism": "Venous Thromboembolism",
    "sex_m": "Male", "sex_f": "Female",
    "alcohol": "Alcohol use", "anemia": "Anaemia",
}, inplace=True)

## 5. Distribution plots

Distributions of the continuous measurements and the binary comorbidities, plus
the per-measurement missing-value flags used to exclude imputed values from the
median (IQR) rows below.

In [ ]:
plt.figure(figsize=(20, 15))
for i, (col, title) in enumerate([
    ("age_at_study_start", "Age"),
    ("bmi_value", "BMI"),
    ("ckd_value", "CKD Value"),
    ("hemoglobin_value", "Hemoglobin Value"),
    ("cholesterol_total", "Total Cholesterol"),
    ("cholesterol_ldl", "LDL Cholesterol"),
    ("cholesterol_hdl", "HDL Cholesterol"),
    ("platelet_count", "Platelet Count"),
    ("systolic_bp", "Systolic Blood Pressure"),
    ("diastolic_bp", "Diastolic Blood Pressure"),
    ("pulse_rate", "Pulse Rate"),
], 1):
    plt.subplot(4, 3, i)
    sns.histplot(pats_comorbidities[col], bins=30, kde=True)
    plt.title(f"{title} Distribution")

plt.tight_layout()
plt.savefig(os.path.join(SUPP_FIGURES_DIR, "fig1_distribution_of_continuous_variables.png"),
            dpi=200, bbox_inches="tight", facecolor="white", transparent=False)
plt.show()

In [ ]:
binary_plot_cols = [
    "Hypertension", "Diabetes", "Cancer", "Liver Disease",
    "Serious Bleeding History", "Alcohol use", "Anaemia",
    "Heart Failure", "Coronary Heart Disease", "Angina", "Myocardial Infarction",
    "Atrial Fibrillation", "Venous Thromboembolism", "Smoking Status",
]

plt.figure(figsize=(20, 15))
for i, col in enumerate(binary_plot_cols, 1):
    plt.subplot(6, 3, i)
    ax = sns.countplot(x=pats_comorbidities[col])
    total = len(pats_comorbidities[col].dropna())
    for p in ax.patches:
        count = int(p.get_height())
        pct = 100 * count / total
        ax.annotate(f"{count}\n({pct:.1f}%)",
                    (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom")
    plt.title(f"{col} distribution")
    plt.xlabel("")
    plt.ylim(0, max(ax.get_ylim()[1], 1.16 * total))
    plt.ylabel("Count")

plt.tight_layout()
plt.savefig(os.path.join(SUPP_FIGURES_DIR, "distribution_of_binary_variables.png"),
            dpi=200, bbox_inches="tight", facecolor="white", transparent=False)
plt.show()

In [ ]:
pats_with_missing_values = pd.read_excel(os.path.join(DERIVED_DIR, "pats_with_missing_comorbidities_values_df.xlsx"), dtype=str)

## 6. Shared table helpers

A single source of truth for the column layout and formatting used by every
characteristic table. Columns are always `Overall`, `Serious Bleeding`,
`Minor Bleeding`, `All Bleeding`. Where a table reports a p-value it always
tests the **primary** outcome — Serious Bleeding vs No Serious Bleeding
(chi-square for categorical/binned variables, Mann–Whitney U for medians) — via
`primary_outcome_masks`, independent of which cohorts are shown as columns.

In [ ]:
COHORT_LABELS = ["Overall", "Serious Bleeding", "Minor Bleeding", "All Bleeding"]


def cohort_masks(df):
    """Ordered dict {cohort label -> boolean mask over df rows}."""
    pid = df["patid"]
    return {
        "Overall": pid.notna(),
        "Serious Bleeding": pid.isin(pats_with_serious_bleeding_events),
        "Minor Bleeding": pid.isin(pats_with_minor_bleeding_events),
        "All Bleeding": pid.isin(pats_with_all_bleeding_events),
    }


def primary_outcome_masks(df):
    """(event, no-event) masks for the PRIMARY (serious) bleeding outcome,
    used for every p-value test regardless of the displayed columns."""
    pid = df["patid"]
    return (pid.isin(pats_with_serious_bleeding_events),
            pid.isin(pats_with_no_serious_bleeding_events))


def format_n_percent(a, total=None):
    """Format a binary count as 'n (pct)'. Accepts either a 0/1 series
    (pct over non-missing) or an explicit (count, total)."""
    if total is None:
        series = pd.Series(a).dropna()
        n_total = len(series)
        n_positive = series.sum()
    else:
        n_positive, n_total = a, total
    if n_total == 0:
        return np.nan
    pct = 100 * n_positive / n_total
    return f"{int(n_positive):,} ({pct:.1f})"


def format_median_iqr(series):
    return f"{series.median():.2f} ({series.quantile(0.25):.2f}–{series.quantile(0.75):.2f})"


def chi2_pvalue(cat_series, event_mask, noevent_mask):
    """Omnibus chi-square p across the categories of cat_series, comparing the
    serious vs no-serious groups. Returns a formatted string or np.nan."""
    try:
        ev = cat_series[event_mask].dropna()
        no = cat_series[noevent_mask].dropna()
        cats = sorted(set(ev.unique()) | set(no.unique()))
        table = np.array([
            [int((ev == c).sum()) for c in cats],
            [int((no == c).sum()) for c in cats],
        ])
        table = table[:, table.sum(axis=0) > 0]  # drop empty categories
        if table.shape[1] < 2 or table.sum() == 0:
            return np.nan
        _, p, _, _ = chi2_contingency(table)
        return "<0.001" if p < 0.001 else f"{p:.3f}"
    except Exception:
        return np.nan


def mwu_pvalue(values, event_mask, noevent_mask):
    """Mann-Whitney U p comparing a continuous variable between the serious and
    no-serious groups."""
    try:
        ev = pd.to_numeric(values[event_mask], errors="coerce").dropna()
        no = pd.to_numeric(values[noevent_mask], errors="coerce").dropna()
        if len(ev) == 0 or len(no) == 0:
            return np.nan
        _, p = mannwhitneyu(ev, no, alternative="two-sided")
        return "<0.001" if p < 0.001 else f"{p:.3f}"
    except Exception:
        return np.nan


# Display labels for the continuous / measurement variables.
PRETTY = {
    "age_at_study_start": "Age, years",
    "bmi_value": "BMI, kg/m\u00b2",
    "ckd_value": "eGFR, mL/min/1.73m\u00b2",
    "hemoglobin_value": "Haemoglobin, g/L",
    "platelet_count": "Platelet count, \u00d710\u2079/L",
    "cholesterol_total": "Total cholesterol, mmol/L",
    "cholesterol_ldl": "LDL cholesterol, mmol/L",
    "cholesterol_hdl": "HDL cholesterol, mmol/L",
    "systolic_bp": "Systolic BP, mmHg",
    "diastolic_bp": "Diastolic BP, mmHg",
    "pulse_rate": "Pulse rate, bpm",
}


def missing_pats(missing_df, flag_col):
    """patids flagged as missing for a measurement, robust to an absent flag."""
    if flag_col in missing_df.columns:
        return missing_df.loc[missing_df[flag_col].astype(str) == "1", "patid"].unique()
    print(f"[warn] '{flag_col}' not found in missing-values export; treating as no missing.")
    return np.array([])

In [ ]:
def binary_table1(df, binary_cols):
    """n (%) with each flag present, per outcome cohort."""
    masks = cohort_masks(df)
    rows = []
    for col in binary_cols:
        row = {"Variable": col}
        for name, mask in masks.items():
            row[name] = format_n_percent(df.loc[mask, col].dropna())
        rows.append(row)
    return pd.DataFrame(rows)[["Variable"] + COHORT_LABELS]


def continuous_table1(df, continuous_cols):
    """Median (IQR) per outcome cohort, excluding per-variable missing values."""
    masks = cohort_masks(df)
    rows = []
    for col in continuous_cols:
        missing = df["patid"].isin(missing_pats(pats_with_missing_values, col))
        row = {"Variable": PRETTY.get(col, col.replace("_", " ").title())}
        for name, mask in masks.items():
            row[name] = format_median_iqr(df.loc[mask & ~missing, col])
        rows.append(row)
    return pd.DataFrame(rows)[["Variable"] + COHORT_LABELS]


def binned_table(df, value_col, median_label, bin_edges, bin_labels,
                 missing_pats=None, include_median=True, unit=None):
    """n (%) per fixed bin of a continuous variable, per outcome cohort, with an
    optional median (IQR) row, a Missing row, and a chi-square p-value."""
    df = df.copy()
    df["_val"] = pd.to_numeric(df[value_col], errors="coerce")
    if missing_pats is None:
        missing_pats = np.array([])
    df["_missing"] = df["patid"].isin(missing_pats)
    df["_cat"] = pd.cut(df["_val"], bins=bin_edges, labels=bin_labels,
                        right=False, include_lowest=True).astype(object)
    df.loc[df["_val"].isna() | df["_missing"], "_cat"] = np.nan

    masks = cohort_masks(df)
    ev, no = primary_outcome_masks(df)
    p = chi2_pvalue(df["_cat"], ev, no)

    rows = []
    if include_median:
        rows.append({
            "Variable": median_label,
            **{name: format_median_iqr(df.loc[mask & ~df["_missing"], "_val"])
               for name, mask in masks.items()},
            "P-value": p,
        })
    for k, label in enumerate(bin_labels):
        rows.append({
            "Variable": label if unit is None else f"{label} {unit}",
            **{name: format_n_percent(int(((df["_cat"] == label) & mask).sum()), int(mask.sum()))
               for name, mask in masks.items()},
            "P-value": ("" if include_median else (p if k == 0 else "")),
        })
    if df["_missing"].any():
        rows.append({
            "Variable": "Missing",
            **{name: format_n_percent(int((df["_missing"] & mask).sum()), int(mask.sum()))
               for name, mask in masks.items()},
            "P-value": "",
        })
    return pd.DataFrame(rows)[["Variable"] + COHORT_LABELS + ["P-value"]]


def quartile_table(df, value_col, median_label, missing_pats=None, unit="mmHg"):
    """Data-driven quartiles (Q1-Q4) of a continuous variable: n (%) per
    quartile per cohort, a median (IQR) row, a Missing row and a Mann-Whitney U
    p-value. Quartile labels show the observed value range."""
    df = df.copy()
    df["_val"] = pd.to_numeric(df[value_col], errors="coerce")
    if missing_pats is None:
        missing_pats = np.array([])
    df["_missing"] = df["patid"].isin(missing_pats)

    masks = cohort_masks(df)
    ev, no = primary_outcome_masks(df)
    p = mwu_pvalue(df["_val"], ev, no)

    df["_quartile"] = pd.Series(np.nan, index=df.index, dtype=object)
    valid = df.loc[~df["_missing"], "_val"].dropna()
    if len(valid):
        df.loc[valid.index, "_quartile"] = pd.qcut(
            valid, q=4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop"
        ).astype(object)

    q_labels = {}
    for q in ["Q1", "Q2", "Q3", "Q4"]:
        vals = df.loc[df["_quartile"] == q, "_val"]
        if len(vals):
            q_labels[q] = f"{q} ({vals.min():.0f}-{vals.max():.0f} {unit})"

    rows = [{
        "Variable": median_label,
        **{name: format_median_iqr(df.loc[mask & ~df["_missing"], "_val"])
           for name, mask in masks.items()},
        "P-value": p,
    }]
    for q in ["Q1", "Q2", "Q3", "Q4"]:
        if q not in q_labels:
            continue
        rows.append({
            "Variable": q_labels[q],
            **{name: format_n_percent(int(((df["_quartile"] == q) & mask).sum()), int(mask.sum()))
               for name, mask in masks.items()},
            "P-value": "",
        })
    rows.append({
        "Variable": "Missing",
        **{name: format_n_percent(int((df["_missing"] & mask).sum()), int(mask.sum()))
           for name, mask in masks.items()},
        "P-value": "",
    })
    return pd.DataFrame(rows)[["Variable"] + COHORT_LABELS + ["P-value"]]


def year_table(df, year_col="calendar_year"):
    """n (%) of patients by calendar year of cohort entry, per outcome cohort,
    with a Missing row."""
    df = df.copy()
    df["_missing"] = df[year_col].isna()
    masks = cohort_masks(df)
    rows = []
    for yr in sorted(df.loc[~df["_missing"], year_col].unique()):
        rows.append({
            "Variable": str(int(yr)),
            **{name: format_n_percent(
                   int(((df[year_col] == yr) & ~df["_missing"] & mask).sum()), int(mask.sum()))
               for name, mask in masks.items()},
        })
    if df["_missing"].any():
        rows.append({
            "Variable": "Missing",
            **{name: format_n_percent(int((df["_missing"] & mask).sum()), int(mask.sum()))
               for name, mask in masks.items()},
        })
    return pd.DataFrame(rows)[["Variable"] + COHORT_LABELS]


def categorical_table(df, cat_col, categories=None):
    """n (%) per category of an already-categorical column, per outcome cohort,
    with a Missing row."""
    df = df.copy()
    df["_missing"] = df[cat_col].isna()
    masks = cohort_masks(df)
    if categories is None:
        categories = sorted(df.loc[~df["_missing"], cat_col].dropna().unique())
    rows = []
    for cat in categories:
        rows.append({
            "Variable": str(cat),
            **{name: format_n_percent(
                   int(((df[cat_col] == cat) & ~df["_missing"] & mask).sum()), int(mask.sum()))
               for name, mask in masks.items()},
        })
    if df["_missing"].any():
        rows.append({
            "Variable": "Missing",
            **{name: format_n_percent(int((df["_missing"] & mask).sum()), int(mask.sum()))
               for name, mask in masks.items()},
        })
    return pd.DataFrame(rows)[["Variable"] + COHORT_LABELS]

## 7. Binary comorbidities

In [ ]:
binary_char_cols = [c for c in [
    "Hypertension", "Diabetes", "Cancer", "Liver Disease", "Serious Bleeding History",
    "Alcohol use", "Anaemia", "Heart Failure", "Coronary Heart Disease", "Angina",
    "Myocardial Infarction", "Atrial Fibrillation", "Venous Thromboembolism",
    "Smoking Status", "Male", "Female",
] if c in pats_comorbidities.columns]

binary_characteristics_df = binary_table1(pats_comorbidities, binary_char_cols)
binary_characteristics_df

## 8. Continuous measurements

In [ ]:
continuous_characteristics_df = continuous_table1(pats_comorbidities, continuous_cols)
continuous_characteristics_df

## 9. Age groups

In [ ]:
age_group_fine_bins = [18, 40, 50, 60, 70, 80, 90, 200]
age_group_fine_labels = ["18-39", "40-49", "50-59", "60-69", "70-79", "80-89", "90+"]

age_group_fine_characteristics_df = binned_table(
    pats_comorbidities,
    value_col="age_at_study_start",
    median_label="Median Age (IQR)",
    bin_edges=age_group_fine_bins,
    bin_labels=age_group_fine_labels,
    include_median=True,
)
age_group_fine_characteristics_df

## 10. Calendar year of cohort entry

In [ ]:
study_start_map = major_doac_exposure.set_index("patid")["study_start_date"]
pats_comorbidities["study_start_date"] = pd.to_datetime(
    pats_comorbidities["patid"].map(study_start_map), errors="coerce"
)
pats_comorbidities["calendar_year"] = pats_comorbidities["study_start_date"].dt.year

calendar_year_characteristics_df = year_table(pats_comorbidities)
calendar_year_characteristics_df

## 11. Systolic blood pressure

In [ ]:
pats_with_missing_sbp_data = missing_pats(pats_with_missing_values, "systolic_bp")

sbp_characteristics_df = quartile_table(
    pats_comorbidities,
    value_col="systolic_bp",
    median_label="Median Systolic BP (IQR)",
    missing_pats=pats_with_missing_sbp_data,
    unit="mmHg",
)
sbp_characteristics_df

## 12. Diastolic blood pressure

In [ ]:
pats_with_missing_dbp_data = missing_pats(pats_with_missing_values, "diastolic_bp")

dbp_characteristics_df = quartile_table(
    pats_comorbidities,
    value_col="diastolic_bp",
    median_label="Median Diastolic BP (IQR)",
    missing_pats=pats_with_missing_dbp_data,
    unit="mmHg",
)
dbp_characteristics_df

## 13. Ethnicity

Extract the earliest recorded ethnicity code per patient from the observation
files, map to the 5-category grouping, then tabulate.

In [ ]:
ethnicity_codes = pd.read_csv(
    os.path.join(CODE_LIST_DIR, "CodeList", "covariate_ethniciity_aurum.txt"), sep="\t", dtype=str
)


def extract_ethnicity_file(file_name):
    with zipfile.ZipFile(OBSERVATION_ZIP, "r") as zip_ref:
        df = pd.read_csv(zip_ref.open(file_name), sep="\t", dtype=str)
    df = df[df["patid"].isin(major_doac_exposure["patid"])]
    df = df[df["medcodeid"].isin(ethnicity_codes["medcodeid"])]
    return df


results = []
if __name__ == "__main__":
    with zipfile.ZipFile(OBSERVATION_ZIP, "r") as zip_ref:
        txt_files = [f for f in zip_ref.namelist() if f.endswith(".txt")]
    with Pool(n_processes) as pool:
        results = pool.map(extract_ethnicity_file, txt_files)

if results:
    ethnicity_data = pd.concat(results, ignore_index=True)
else:
    ethnicity_data = pd.DataFrame(columns=["patid", "medcodeid", "obsdate"])

ethnicity_data["ethnicity"] = ethnicity_data["medcodeid"].map(
    ethnicity_codes.set_index("medcodeid")["eth5"]
)
ethnicity_data.sort_values(by=["patid", "obsdate"], inplace=True)
ethnicity_data.drop_duplicates(subset=["patid"], keep="first", inplace=True)
ethnicity_data = ethnicity_data[["patid", "ethnicity"]].reset_index(drop=True)
ethnicity_data

In [ ]:
pats_comorbidities["ethnicity"] = pats_comorbidities["patid"].map(
    ethnicity_data.set_index("patid")["ethnicity"]
)
ethnicity_characteristics_df = categorical_table(pats_comorbidities, cat_col="ethnicity")
ethnicity_characteristics_df

In [ ]:
pats_comorbidities["ethnicity"].value_counts()

## 14. Polypharmacy

Assign each patient to a mutually-exclusive drug-count group by removing anyone
already in a higher group, using the pre-computed patient lists.

In [ ]:
polypharmacy_pats = pd.read_excel(os.path.join(DERIVED_DIR, "polypharmacy_pats_df.xlsx"), dtype=str)

zero_drug_pats = polypharmacy_pats["zero_drugs_pats"].dropna().unique()
one_drug_pats = polypharmacy_pats["one_drug_pats"].dropna().unique()
two_drug_pats = polypharmacy_pats["two_drugs_pats"].dropna().unique()
three_drug_pats = polypharmacy_pats["three_drugs_pats"].dropna().unique()
four_drug_pats = polypharmacy_pats["four_drugs_pats"].dropna().unique()
five_drug_pats = polypharmacy_pats["five_or_more_drugs_pats"].dropna().unique()

# Make the groups mutually exclusive (highest group wins).
five_or_more_drugs_pats = set(five_drug_pats)
four_drugs_pats = set(four_drug_pats) - five_or_more_drugs_pats
three_drugs_pats = set(three_drug_pats) - five_or_more_drugs_pats - four_drugs_pats
two_drugs_pats = set(two_drug_pats) - five_or_more_drugs_pats - four_drugs_pats - three_drugs_pats
one_drug_pats = set(one_drug_pats) - five_or_more_drugs_pats - four_drugs_pats - three_drugs_pats - two_drugs_pats
zero_drug_pats = (set(zero_drug_pats) - five_or_more_drugs_pats - four_drugs_pats
                  - three_drugs_pats - two_drugs_pats - one_drug_pats)

for label, group in [("zero", zero_drug_pats), ("one", one_drug_pats),
                     ("two", two_drugs_pats), ("three", three_drugs_pats),
                     ("four", four_drugs_pats), ("five or more", five_or_more_drugs_pats)]:
    print(f"Patients with {label} drug(s): {len(group):,}")
print(f"Total: {len(zero_drug_pats) + len(one_drug_pats) + len(two_drugs_pats) + len(three_drugs_pats) + len(four_drugs_pats) + len(five_or_more_drugs_pats):,}")

In [ ]:
polypharmacy_map = {}
polypharmacy_map.update(dict.fromkeys(zero_drug_pats, "0 Drugs"))
polypharmacy_map.update(dict.fromkeys(one_drug_pats, "1 Drug"))
polypharmacy_map.update(dict.fromkeys(two_drugs_pats, "2 Drugs"))
polypharmacy_map.update(dict.fromkeys(three_drugs_pats, "3 Drugs"))
polypharmacy_map.update(dict.fromkeys(four_drugs_pats, "4 Drugs"))
polypharmacy_map.update(dict.fromkeys(five_or_more_drugs_pats, "5+ Drugs"))

pats_comorbidities["polypharmacy"] = pats_comorbidities["patid"].map(polypharmacy_map)
pats_comorbidities["polypharmacy"].value_counts()

In [ ]:
polypharmacy_order = ["0 Drugs", "1 Drug", "2 Drugs", "3 Drugs", "4 Drugs", "5+ Drugs"]
polypharmacy_characteristics_df = categorical_table(
    pats_comorbidities,
    cat_col="polypharmacy",
    categories=[c for c in polypharmacy_order
                if c in pats_comorbidities["polypharmacy"].dropna().unique()],
)
polypharmacy_characteristics_df

## 15. Proton-pump inhibitor (PPI) use

Flag patients with any PPI prescription in the 180 days before study entry, then
tabulate as a binary characteristic.

In [ ]:
prod_dict = pd.read_csv(PRODUCT_DICT, sep="\t", dtype=str)
ppi_drug_codes = prod_dict[prod_dict["drug_class"].str.contains("Proton Pump Inhibitor", case=False, na=False)]
ppi_drug_codes = ppi_drug_codes[
    ~ppi_drug_codes["DrugSubstanceName"].str.contains("/", case=False, na=False)
]["ProdCodeId"].unique().tolist()

zip_path = DRUG_ISSUE_ZIP


def extract_ppi_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep="\t", dtype=str)

    df["issuedate"] = pd.to_datetime(df["issuedate"], errors="coerce")
    df = df[
        df["prodcodeid"].isin(ppi_drug_codes)
        & df["patid"].isin(serious_doac_exposure["patid"])
    ].copy()
    df["study_start_date"] = pd.to_datetime(
        df["patid"].map(serious_doac_exposure.set_index("patid")["study_start_date"]),
        errors="coerce",
    )
    df = df[
        (df["issuedate"] >= df["study_start_date"] - pd.Timedelta(days=180))
        & (df["issuedate"] < df["study_start_date"])
    ]
    return df["patid"].unique().tolist()


ppi_pats = set()
if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith(".txt")]
    with Pool(n_processes) as pool:
        all_results = pool.map(extract_ppi_file, txt_files)
    ppi_pats = set(p for sublist in all_results for p in sublist)

In [ ]:
pats_comorbidities["PPI use"] = pats_comorbidities["patid"].isin(ppi_pats).astype(int)

ppi_characteristics_df = binary_table1(pats_comorbidities, ["PPI use"])
ppi_characteristics_df

## 16. Exposure & follow-up

Follow-up time (`study_end_date - study_start_date`) comes from the analytic
cohort; `study_end_date` already reflects censoring at the earliest of outcome,
death, transfer-out, last collection or study end. The serious/minor/all frames
share the same patients and window and differ only in which event is populated.

In [ ]:
fu = serious_doac_exposure[["patid", "study_start_date", "study_end_date"]].copy()
fu["follow_up_days"] = (fu["study_end_date"] - fu["study_start_date"]).dt.days
fu = fu[fu["follow_up_days"] >= 0]

total_py = fu["follow_up_days"].sum() / 365.25
n_cohort = fu["patid"].nunique()


def _n_events(frame):
    return int(frame.loc[frame["event_date"].notna(), "patid"].nunique())


def _rate(n):
    return 1000 * n / total_py if total_py else np.nan


n_serious = _n_events(serious_doac_exposure)
n_minor = _n_events(minor_doac_exposure)
n_all = _n_events(all_doac_exposure)

follow_up_summary = pd.DataFrame([
    {"Metric": "Patients", "Value": f"{n_cohort:,}"},
    {"Metric": "Total follow-up (person-years)", "Value": f"{total_py:,.1f}"},
    {"Metric": "Median follow-up, days (IQR)", "Value": format_median_iqr(fu["follow_up_days"])},
    {"Metric": "Median follow-up, years (IQR)", "Value": format_median_iqr(fu["follow_up_days"] / 365.25)},
    {"Metric": "Serious bleeding events (rate /1000 PY)", "Value": f"{n_serious:,} ({_rate(n_serious):.2f})"},
    {"Metric": "Minor bleeding events (rate /1000 PY)", "Value": f"{n_minor:,} ({_rate(n_minor):.2f})"},
    {"Metric": "All bleeding events (rate /1000 PY)", "Value": f"{n_all:,} ({_rate(n_all):.2f})"},
])
follow_up_summary